In [0]:
from pyspark.sql import functions as F

SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@ecommercedatalakebp01.dfs.core.windows.net"

In [0]:
df_returns = spark.read.format("delta").load(f"{SILVER_PATH}/returns")

print("total returns:", df_returns.count())
print("distinct order_id:", df_returns.select("order_id").distinct().count())

In [0]:
df_customers=spark.read.format("delta").load(f"{SILVER_PATH}/customers")
print(df_customers.columns)

In [0]:
df_customers=df_customers.select("customer_id","customer_name","email","city","state","country","signup_date")

In [0]:
special_rows=[("UNKNOWN","Unknown customer","Unknown","Unknown","Unknown","India",None),
              ("GUEST","Guest visitor","Unknown","Unknown","Unknown","India",None)]
df_special= spark.createDataFrame(special_rows,schema=df_customers.schema)
df_special.display()

In [0]:
dim_customers=df_customers.unionByName(df_special)

In [0]:
print("rows:", dim_customers.count())

In [0]:
assert dim_customers.select("customer_id").distinct().count() == dim_customers.count(),"Duplicates have been found"

In [0]:
dim_customers=dim_customers.withColumn("_processed_at", F.current_timestamp())

In [0]:
dim_customers.write.mode("overwrite").format("delta").save(f"{GOLD_PATH}/dim_customers")

In [0]:
check=spark.read.format("delta").load(f"{GOLD_PATH}/dim_customers")
print("rows:",check.count())
print("columns:", check.columns)